## 1. Partindo de uma página bruta

Nesta prática, vamos usar uma página fictícia do livro `Integrações Resilientes`. O objetivo não é extrair estrutura do documento ainda. Vamos tratar a página como uma sequência contínua de caracteres e observar o que acontece quando o corte segue apenas tamanho e overlap.

In [1]:
page_metadata = {
    "book_title": "Integracoes Resilientes: webhooks, filas e retentativas na pratica",
    "edition": "2a edicao",
    "chapter": "Capitulo 4 - Webhooks em producao",
    "section": "4.3 Timeouts e retentativas",
    "page_start": 118,
    "page_end": 119,
}

raw_page = """
# 4.3 Timeouts e retentativas em webhooks

Quando um provedor envia um webhook, ele espera uma resposta rapida do consumidor.
Se a conexao expira antes de receber confirmacao, o provedor nao sabe se o evento
falhou antes de chegar, se foi processado parcialmente ou se a resposta se perdeu
no caminho de volta.

Por isso, timeout nao deve ser tratado como erro definitivo. Em geral, ele entra
na mesma familia de falhas temporarias: o provedor registra a tentativa, agenda
uma nova entrega e preserva o mesmo identificador de evento para permitir
deduplicacao no consumidor.

| status | significado | acao recomendada |
| 200 | evento recebido e persistido | encerrar entrega |
| 408 | consumidor nao respondeu dentro do limite | reagendar com backoff |
| 429 | consumidor pediu reducao de ritmo | aplicar espera maior |
| 500 | falha temporaria no consumidor | tentar novamente com limite |

Um payload de entrega deve carregar informacao suficiente para tornar a
retentativa segura:

```json
{
  "event_id": "evt_8f3a",
  "type": "invoice.paid",
  "attempt": 3,
  "occurred_at": "2026-08-11T14:20:00Z"
}
```

O consumidor deve gravar `event_id` antes de executar efeitos irreversiveis.
Se a mesma entrega aparecer de novo depois de um timeout, a aplicacao reconhece
o identificador e evita cobrar, enviar e-mail ou baixar estoque duas vezes.
""".strip()

print(f"fonte: {page_metadata['book_title']}")
print(f"trecho: {page_metadata['chapter']} > {page_metadata['section']}")
print(f"paginas: {page_metadata['page_start']}-{page_metadata['page_end']}")
print(f"caracteres no texto bruto: {len(raw_page)}")
print(raw_page[:420] + "...")

fonte: Integracoes Resilientes: webhooks, filas e retentativas na pratica
trecho: Capitulo 4 - Webhooks em producao > 4.3 Timeouts e retentativas
paginas: 118-119
caracteres no texto bruto: 1343
# 4.3 Timeouts e retentativas em webhooks

Quando um provedor envia um webhook, ele espera uma resposta rapida do consumidor.
Se a conexao expira antes de receber confirmacao, o provedor nao sabe se o evento
falhou antes de chegar, se foi processado parcialmente ou se a resposta se perdeu
no caminho de volta.

Por isso, timeout nao deve ser tratado como erro definitivo. Em geral, ele entra
na mesma familia de falhas ...


## 2. Estimando tokens por caracteres

Nesta primeira estratégia, não vamos carregar um modelo de embedding nem um tokenizer real. Vamos usar uma heurística comum para planejamento rápido: aproximadamente 1 token a cada 4 caracteres. Isso é imperfeito, mas suficiente para mostrar o comportamento do tamanho fixo.

In [2]:
import math

CHARS_PER_TOKEN = 4
MAX_CHARS = 720
OVERLAP_CHARS = 160

def estimate_tokens(text):
    return math.ceil(len(text) / CHARS_PER_TOKEN)

print(f"heuristica: 1 token ~= {CHARS_PER_TOKEN} caracteres")
print(f"limite por chunk: {MAX_CHARS} caracteres (~{estimate_tokens('x' * MAX_CHARS)} tokens)")
print(f"overlap: {OVERLAP_CHARS} caracteres (~{estimate_tokens('x' * OVERLAP_CHARS)} tokens)")
print(f"texto bruto estimado: {estimate_tokens(raw_page)} tokens")

heuristica: 1 token ~= 4 caracteres
limite por chunk: 720 caracteres (~180 tokens)
overlap: 160 caracteres (~40 tokens)
texto bruto estimado: 336 tokens


## 3. Criando chunks por tamanho fixo com overlap

O algoritmo abaixo só conhece três coisas: texto, tamanho máximo e overlap. Ele não sabe o que é tabela, payload, heading ou nota. Essa ignorância é a característica central da estratégia.

In [3]:
def chunk_by_fixed_size(text, max_chars, overlap_chars):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + max_chars, len(text))
        chunk_text = text[start:end].strip()
        chunks.append({
            "chunk_id": f"fixed-{len(chunks) + 1:02d}",
            "text": chunk_text,
            "char_start": start,
            "char_end": end,
            "estimated_tokens": estimate_tokens(chunk_text),
        })

        if end == len(text):
            break

        start = end - overlap_chars

    return chunks

chunks = chunk_by_fixed_size(raw_page, MAX_CHARS, OVERLAP_CHARS)

print(f"total de chunks: {len(chunks)}")
for chunk in chunks:
    print(
        f"{chunk['chunk_id']} | chars {chunk['char_start']}-{chunk['char_end']} "
        f"| ~{chunk['estimated_tokens']} tokens"
    )

total de chunks: 3
fixed-01 | chars 0-720 | ~180 tokens
fixed-02 | chars 560-1280 | ~180 tokens
fixed-03 | chars 1120-1343 | ~56 tokens


## 4. Inspecionando fronteiras e repetição

Agora olhamos para o texto. O corte por tamanho não respeita fronteiras de sentido: ele pode começar no meio de uma tabela, de uma frase ou de uma palavra. O overlap aparece como texto repetido entre chunks vizinhos.

In [4]:
import re

def compact(text, size=210):
    return re.sub(r"\s+", " ", text).strip()[:size]

for chunk in chunks:
    print(chunk["chunk_id"])
    print(compact(chunk["text"]))
    print()

print("overlap entre fixed-01 e fixed-02:")
print(raw_page[chunks[1]["char_start"]:chunks[0]["char_end"]].replace("\n", " "))

fixed-01
# 4.3 Timeouts e retentativas em webhooks Quando um provedor envia um webhook, ele espera uma resposta rapida do consumidor. Se a conexao expira antes de receber confirmacao, o provedor nao sabe se o evento fal

fixed-02
no consumidor. | status | significado | acao recomendada | | 200 | evento recebido e persistido | encerrar entrega | | 408 | consumidor nao respondeu dentro do limite | reagendar com backoff | | 429 | consumido

fixed-03
dor deve gravar `event_id` antes de executar efeitos irreversiveis. Se a mesma entrega aparecer de novo depois de um timeout, a aplicacao reconhece o identificador e evita cobrar, enviar e-mail ou baixar estoqu

overlap entre fixed-01 e fixed-02:
no consumidor.  | status | significado | acao recomendada | | 200 | evento recebido e persistido | encerrar entrega | | 408 | consumidor nao respondeu dentro do


## 5. Anexando metadados

Mesmo quando a estratégia de corte é simples, cada chunk precisa carregar origem e posição. Sem metadados, o sistema recupera texto, mas perde rastreabilidade.

In [5]:
metadata_chunks = []

for chunk in chunks:
    metadata = {
        **page_metadata,
        "chunk_id": chunk["chunk_id"],
        "strategy": "fixed_size_with_overlap",
        "char_start": chunk["char_start"],
        "char_end": chunk["char_end"],
        "estimated_tokens": chunk["estimated_tokens"],
    }
    metadata_chunks.append({"text": chunk["text"], "metadata": metadata})

for item in metadata_chunks:
    metadata = item["metadata"]
    print(
        f"{metadata['chunk_id']} | {metadata['strategy']} | "
        f"pagina {metadata['page_start']}-{metadata['page_end']} | "
        f"chars {metadata['char_start']}-{metadata['char_end']} | "
        f"~{metadata['estimated_tokens']} tokens"
    )

fixed-01 | fixed_size_with_overlap | pagina 118-119 | chars 0-720 | ~180 tokens
fixed-02 | fixed_size_with_overlap | pagina 118-119 | chars 560-1280 | ~180 tokens
fixed-03 | fixed_size_with_overlap | pagina 118-119 | chars 1120-1343 | ~56 tokens


## 6. Recuperando um candidato

A busca abaixo é simples de propósito. Ela só conta termos da query encontrados em cada chunk. O objetivo não é ensinar ranking lexical de novo; é observar qual unidade recuperável o corte por tamanho fixo oferece para uma pergunta sobre timeout em webhooks.

In [6]:
import unicodedata

STOPWORDS = {"como", "com", "em", "de", "o", "a", "os", "as", "um", "uma"}

def normalize(text):
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    return "".join(char for char in text if not unicodedata.combining(char))

def tokenize(text):
    terms = re.findall(r"[a-z0-9]+", normalize(text))
    return [term for term in terms if len(term) >= 4 and term not in STOPWORDS]

def search(query, chunks):
    query_terms = set(tokenize(query))
    ranking = []

    for item in chunks:
        chunk_terms = tokenize(item["text"])
        matches = sorted(query_terms.intersection(chunk_terms))
        score = sum(chunk_terms.count(term) for term in matches)
        ranking.append((score, matches, item))

    return sorted(ranking, key=lambda result: result[0], reverse=True)

query = "como lidar com timeout em webhooks?"
results = search(query, metadata_chunks)

print(f"query: {query}")
for position, (score, matches, item) in enumerate(results, start=1):
    metadata = item["metadata"]
    matched_terms = ", ".join(matches) if matches else "sem termos da query"
    print(f"#{position} {metadata['chunk_id']} score={score} ({matched_terms})")

best = results[0][2]
print()
print("melhor candidato:")
print(best["metadata"]["chunk_id"])
print(compact(best["text"], 360))

query: como lidar com timeout em webhooks?
#1 fixed-01 score=2 (timeout, webhooks)
#2 fixed-02 score=1 (timeout)
#3 fixed-03 score=1 (timeout)

melhor candidato:
fixed-01
# 4.3 Timeouts e retentativas em webhooks Quando um provedor envia um webhook, ele espera uma resposta rapida do consumidor. Se a conexao expira antes de receber confirmacao, o provedor nao sabe se o evento falhou antes de chegar, se foi processado parcialmente ou se a resposta se perdeu no caminho de volta. Por isso, timeout nao deve ser tratado como erro d


O tamanho fixo foi fácil de implementar e produziu chunks previsíveis. O preço aparece nas fronteiras: a tabela começa dentro de um chunk, o payload pode ficar separado da explicação e o último chunk começa no meio de uma palavra. O overlap ajuda a carregar contexto vizinho, mas também duplica texto e aumenta a quantidade de conteúdo indexado.